# Class Project - What Stories Do Data Tell? Analyzing and Interpreting Real-World Information
### Thierry Laguerre
### Course Data Science Spring 2026

In [ ]:
pip install ucimlrepo

## Data Acquisition
Each group should download and import the chosen dataset into their preferred data analysis environment (e.g., Google Colab).

I decided to use the Bank Marketing, Published on 2/13/2012

"The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y)."

In [ ]:
from ucimlrepo import fetch_ucirepo

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

print(bank_marketing.metadata)

print(bank_marketing.variables)


{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

## Preliminary Data Analysis:

Begin by providing a brief overview of the dataset, including the number of rows and columns, data types, and a description of each variable. Check for missing data and handle it appropriately (e.g., removal). Identify and address any data quality issues, such as outliers or inconsistencies.

In [ ]:
df = pd.concat([X, y], axis=1)

In [ ]:
import pandas as pd
df = pd.concat([X, y], axis=1)
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded with 45211 rows and 17 columns.


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [ ]:
print(df.isnull().sum())

age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
y                  0
dtype: int64


In [ ]:
df_cleaned = df.dropna(subset=['job', 'education']).copy()

df_cleaned['poutcome'] = df_cleaned['poutcome'].fillna('unknown_outcome')
df_cleaned['contact'] = df_cleaned['contact'].fillna('unknown_contact')

## Exploratory Data Analysis (EDA)

Perform a comprehensive EDA, including summary statistics, distribution analysis, and correlation analysis.
Identify interesting patterns, trends, or relationships within the data.

In [ ]:
df_cleaned.describe()

,age,balance,day_of_week,duration,campaign,pdays,previous
count,43193.000000,43193.000000,43193.000000,43193.000000,43193.000000,43193.000000,43193.000000
mean,40.764082,1354.027342,15.809414,258.323409,2.758178,40.404070,0.584863
std,10.512640,3042.103625,8.305970,258.162006,3.063987,100.420624,2.332672
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,71.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,442.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1412.000000,21.000000,318.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,58.000000,871.000000,275.000000


In [ ]:
df_cleaned.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown_contact,5,may,261,1,-1,0,unknown_outcome,no
1,44,technician,single,secondary,no,29,yes,no,unknown_contact,5,may,151,1,-1,0,unknown_outcome,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown_contact,5,may,76,1,-1,0,unknown_outcome,no
5,35,management,married,tertiary,no,231,yes,no,unknown_contact,5,may,139,1,-1,0,unknown_outcome,no
6,28,management,single,tertiary,no,447,yes,yes,unknown_contact,5,may,217,1,-1,0,unknown_outcome,no


In [ ]:
print(f"Before Cleaning shape: {df.shape}")
print(f"After Cleaning shape: {df_cleaned.shape}")

Before Cleaning shape: (45211, 17)
After Cleaning shape: (43193, 17)


In [ ]:
import plotly.express as px

In [ ]:
job_balance = df_cleaned.groupby(['job', 'y'])['balance'].mean().reset_index()

fig = px.bar(job_balance,
             x='job',
             y='balance',
             color='y',
             barmode='group',
             title='Bar Chart showing mean Bank Balance by Job Type and Subscription Status',
             labels={'balance': 'Mean Balance (in Euros)', 'job': 'Job Sector', 'y': 'Subscribed?'},
             template='plotly_white')

fig.show()

In [ ]:
y_counts = df_cleaned['y'].value_counts().reset_index()
y_counts.columns = ['Subscribed', 'Count']

fig = px.pie(y_counts,
             values='Count',
             names='Subscribed',
             title='Percentage of People who Agreed/Subscribed to the Fixed term Deposit',
             color_discrete_sequence=['#EF553B', '#008000'],
             hole=0.4)

fig.update_traces(textinfo='percent+label')
fig.show()

In [ ]:
import plotly.express as px

fig = px.box(df_cleaned,
             x="housing",
             y="balance",
             color="housing",
             title="Box Plot showing Comparisons of Bank Balances by Housing Loan Status",
             labels={"housing": "Has Housing Loan?", "balance": "Yearly Balance (€)"},
             points=False)

fig.update_yaxes(range=[-1000, 5000])

fig.show()

## Based on your initial exploration, propose a research question you'd like to answer.

- The research question for this project: Is there a significant difference in the mean bank balance between clients who have a housing loan and those who do not?

## State your null and alternative hypotheses clearly. For the above question:

- The Null Hypostheis for this research question is that the mean bank balance is the same for those with and without housing loans.

- The Alternative Hyposthesis suggests that the mean bank balance is significantly different between the two groups.

In [ ]:
from scipy import stats

housing_group = df_cleaned[df_cleaned['housing'] == 'yes']['balance']
no_housing_group = df_cleaned[df_cleaned['housing'] == 'no']['balance']

result = stats.ttest_ind(housing_group, no_housing_group, equal_var=False)

if result.pvalue < 0.05:
  print("Evidence to reject null hypothesis")
else:
  print("We cannot reject the null hypothesis. Difference is likely due to chance.")

print(f"Mean Balance for those with housing loan is {housing_group.mean():.2f} (in euros)")
print(f"Mean Balance for those without loan is {no_housing_group.mean():.2f} (in euros)")
print("T stat", result.statistic)
print(f"P-value: {result.pvalue:.10f}")
print(f"Degrees of Freedom: {result.df}")

Evidence to reject null hypothesis
Mean Balance for those with housing loan is 1174.14 (in euros)
Mean Balance for those without loan is 1585.22 (in euros)
T stat -13.34150132247087
P-value: 0.0000000000
Degrees of Freedom: 31822.671411094245


## Based on the p-value and a significance level (e.g., 0.05), determine if you should reject the null hypothesis.

- Based on the P-Value being less than the 0.05 threshold We Reject the Null Hypothesis. As shown in the visualizations there is a difference between the bank balances for people with mortgages and those without.

## Clearly explain the practical implications of your findings. For example, if there is a significant difference in wine quality scores between red and white wines, what might this imply for wine producers or consumers?

- Based on the fact that people with mortgages have less in the bank and there being a low probability that the results were an accident, the bank should pivot their marketing. The bank should make people without a housing loan their target audience for Term Deposit marketing. It makes sense because people with mortgages have to allocate income towards it every month meaning they have less savings. Those without mortgages most likely have extra money they can use to invest in a fixed term investment.


# Report Generation:

### Write a detailed report in slides presentation format that includes:
- A brief introduction to the dataset and its context.

The dataset I chose for this project is the Portugese Bank Marketing Dataset. The dataset contained financial profiles of clients and the goal was to determine how to better predict who would be willing to accept or invest in a fixed term deposit. The important variable was the y which is the outcome of whether they subscribed or bought into the fixed term deposit. This would help banks identify and better market to clients with available or extra cash laying around that they can use to make profits.

- A description of the data preprocessing steps.

First steps was merging features and targets into a single dataframe because of the way I fetched the data. Moreover, I dropped rows with missing job and education to better indentify the group banks should target. I replaced null values in the contact and poutcome with unknown string. I did this because a large number of the clients were new and I did not want to lose all of that data.

- Summarized findings from the EDA, highlighting key insights.

From looking at the statistics I learned that the average age of the clients in Portugese banks are around the age of 40. It makes sense considering younger people may prefer more high risk investments or have not accumulated as much savings. The average balance was less than 2000 in euros. Moreover, I indetified outliers such as someone carrying a negative balance of -8019 in euros

- Interpretation of visualizations and what they reveal about the data.

Visualizations were done with plotly. I was curious to how successful the marketing team was in getting commitments and the plotly pie chart showed around 11.7 percent of people. This shows that there was a lot of wasted effort which makes finding people who will agree to fixed term investment is important.  Morever I did a bar plot showing bank balances and job type and it turns out the people with the most cash readily available are retirees. Finally, a box plot revealed that people without mortgages on average have higher bank balances.

- Any conclusions or recommendations drawn from the hypothesis testing.

My hypotheis testing questioned whether there is a difference in bak balances between clients with housing loans and those without. I conducted calculations using two sample ttest ensuring groups were independent and realized a hidden correlation. I rejected the null hypothesis due to the p-value being lower than the 0.05 threshold. A clear difference of around 411 in euros is present between the two groups bank balnce.

- A list of references for any external sources used.

Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306.